# YOLOv8 Hand Detector — Training on Colab (with W&B)

**Capstone CV 2025.2** — Stage 1 của pipeline (sau notebook này tới classifier).

Train YOLOv8-nano cho 1 lớp `hand` trên dataset Roboflow Universe. W&B log đầy đủ qua tích hợp Ultralytics.

## Yêu cầu

1. **GPU**: Runtime → Change runtime type → **T4 GPU** (free).
2. **W&B API key**: https://wandb.ai/authorize
3. **Roboflow API key**: https://app.roboflow.com/settings/api (tạo account free).
4. (Tùy chọn) Mount Google Drive.

## Dataset chiến lược

HaGRID (~700 GB) quá lớn cho Colab → dùng **Roboflow Universe**, có nhiều dataset hand-detection nhỏ hơn (~5k ảnh, đã ở format YOLO):

- Default: workspace `roboflow-100`, project `hand-detection-vmkj0` — ~9k ảnh, đã split
- Alternative: `EgoHands`, `OXFORD Hand Dataset` (cần convert format)
- Nếu bạn muốn tự host HaGRID subset → dùng cell tùy chọn ở mục 3.

## W&B sẽ log gì

- **Auto** (Ultralytics callback): box loss, cls loss, dfl loss, lr, mAP@0.5, mAP@0.5:0.95, precision, recall mỗi epoch
- **Validation samples**: ảnh + bbox dự đoán mỗi N epoch
- **System**: GPU util, memory
- **Final**: confusion matrix, PR curve, F1 curve, validation predictions table
- **Model artifact**: best.pt + last.pt (versioned)

Thời gian: 60–90 phút trên T4 cho 50 epochs với ~9k images.

## 0 · GPU + Mount Drive

In [ ]:
import torch, os
print(f'PyTorch: {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    !nvidia-smi -L

USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/signlang'
else:
    SAVE_DIR = '/content/signlang_out'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Output: {SAVE_DIR}')

## 1 · Cài Ultralytics + W&B + Roboflow

In [ ]:
!pip install -q ultralytics wandb roboflow

import wandb
from ultralytics import YOLO, settings

wandb.login()

# Bật tích hợp Ultralytics → W&B (auto-log mọi metric trong training)
settings.update({'wandb': True})
print('Ultralytics + W&B integration enabled.')

## 2 · Tải dataset từ Roboflow Universe

Lấy API key tại https://app.roboflow.com/settings/api (free tier OK).

In [ ]:
import getpass
from roboflow import Roboflow

if not os.environ.get('ROBOFLOW_API_KEY'):
    os.environ['ROBOFLOW_API_KEY'] = getpass.getpass('Roboflow API key: ')

# Default project: roboflow-100/hand-detection — ~9k ảnh hand annotations.
# Bạn có thể đổi sang dataset khác bằng cách chỉnh 3 dòng dưới đây.
RF_WORKSPACE = 'roboflow-100'
RF_PROJECT   = 'hand-detection-vmkj0'
RF_VERSION   = 2

rf = Roboflow(api_key=os.environ['ROBOFLOW_API_KEY'])
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download('yolov8', location='/content/hand_dataset')

DATA_YAML = f'{dataset.location}/data.yaml'
print(f'Dataset YAML: {DATA_YAML}')
with open(DATA_YAML) as f:
    print(f.read())

### 2b · (Tùy chọn) Dùng HaGRID subset thay vì Roboflow

Bỏ qua cell này nếu đã chạy mục 2. Cell này tải HaGRID-mini (~250 MB, 1k ảnh) và convert sang YOLO format. Phù hợp khi bạn không muốn tạo Roboflow account.

```python
# Uncomment to use:
# !pip install -q huggingface_hub
# from huggingface_hub import snapshot_download
# snapshot_download(repo_id='Ar4ikov/hagrid-mini', repo_type='dataset', local_dir='/content/hagrid')
# (Xem repo gốc HaGRID để convert annotations sang YOLO format.)
```

## 3 · Hyperparameters

In [ ]:
config = {
    'model_variant': 'yolov8n.pt',     # 'yolov8n' (3.2M) | 'yolov8s' (11.2M) | 'yolov8m' (25.9M)
    'epochs': 50,
    'batch': 16,
    'imgsz': 640,
    'patience': 10,                     # early stopping
    'optimizer': 'SGD',                 # 'SGD' | 'Adam' | 'AdamW'
    'lr0': 0.01,
    'lrf': 0.01,                         # final lr = lr0 * lrf
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'cos_lr': True,
    'augment': True,
    'mosaic': 1.0,
    'mixup': 0.0,
    'amp': True,
    'workers': 4,
    'seed': 11711,
    'project_name': 'signlang-detector',
    'run_name': 'yolov8n-roboflow-handdet',
}
print(config)

## 4 · Train với W&B logging

Ultralytics tự log lên W&B khi `settings['wandb']=True` ở cell 1. Chỉ cần `wandb.init()` trước khi gọi `model.train()`.

In [ ]:
run = wandb.init(
    project=config['project_name'],
    name=config['run_name'],
    config=config,
    tags=['detector', 'yolov8', 'hand'],
    notes='YOLOv8-nano hand detector for ASL pipeline stage 1',
)

# Try to use the modern callback (preferred). If not available (older wandb)
# or if it crashes due to the known `RANK undefined` bug in wandb 0.18+/Ultralytics 8.3+,
# fall back to the built-in Ultralytics→W&B integration (enabled at cell 1).
model = YOLO(config['model_variant'])
try:
    from wandb.integration.ultralytics import add_wandb_callback
    add_wandb_callback(model, enable_model_checkpointing=True)
    print('Using add_wandb_callback (auto-logs validation predictions table).')
except (ImportError, NameError, Exception) as e:
    print(f'Skipping add_wandb_callback ({type(e).__name__}: {e}); built-in Ultralytics→W&B integration still active.')

results = model.train(
    data=DATA_YAML,
    epochs=config['epochs'],
    batch=config['batch'],
    imgsz=config['imgsz'],
    patience=config['patience'],
    optimizer=config['optimizer'],
    lr0=config['lr0'],
    lrf=config['lrf'],
    momentum=config['momentum'],
    weight_decay=config['weight_decay'],
    warmup_epochs=config['warmup_epochs'],
    cos_lr=config['cos_lr'],
    augment=config['augment'],
    mosaic=config['mosaic'],
    mixup=config['mixup'],
    amp=config['amp'],
    workers=config['workers'],
    seed=config['seed'],
    project='/content/runs',
    name=config['run_name'],
    exist_ok=True,
)

BEST_PT = f"/content/runs/{config['run_name']}/weights/best.pt"
LAST_PT = f"/content/runs/{config['run_name']}/weights/last.pt"
print(f'Best weights: {BEST_PT}')

## 5 · Validation cuối cùng — log thêm metrics chi tiết

In [ ]:
best = YOLO(BEST_PT)

# Re-attach to W&B (Ultralytics built-in integration auto-finishes the run after training).
if wandb.run is None:
    try:
        wandb.init(project=run.project, id=run.id, resume='allow')
        print(f'Resumed W&B run {run.id}')
    except (NameError, AttributeError):
        wandb.init(project=config['project_name'],
                   name=config['run_name'] + '-eval',
                   config=config, tags=['eval'])
        print('Created new eval run (could not resume).')

metrics = best.val(data=DATA_YAML, imgsz=config['imgsz'], batch=config['batch'], split='val')

# Log final metrics into W&B summary so chúng nổi bật ở đầu run page
wandb.summary['final/mAP50']      = float(metrics.box.map50)
wandb.summary['final/mAP50_95']   = float(metrics.box.map)
wandb.summary['final/precision']  = float(metrics.box.mp)
wandb.summary['final/recall']     = float(metrics.box.mr)

print(f"mAP@0.5      = {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 = {metrics.box.map:.4f}")
print(f"Precision    = {metrics.box.mp:.4f}")
print(f"Recall       = {metrics.box.mr:.4f}")

## 6 · Sample predictions trên test set

In [ ]:
import glob, random
from PIL import Image

# Re-attach if needed
if wandb.run is None:
    try:
        wandb.init(project=run.project, id=run.id, resume='allow')
    except (NameError, AttributeError):
        pass

# Roboflow datasets có sẵn split test (hoặc valid)
test_dir = f'{dataset.location}/test/images'
if not os.path.isdir(test_dir):
    test_dir = f'{dataset.location}/valid/images'

samples = random.sample(glob.glob(f'{test_dir}/*.jpg') + glob.glob(f'{test_dir}/*.png'), min(12, 200))
predictions_table = wandb.Table(columns=['image', 'num_detections', 'max_confidence'])
for p in samples:
    res = best.predict(source=p, conf=0.25, verbose=False)[0]
    plotted = res.plot()    # numpy BGR with bboxes drawn
    plotted_rgb = plotted[..., ::-1]
    n = len(res.boxes); max_conf = float(res.boxes.conf.max()) if n > 0 else 0.0
    predictions_table.add_data(wandb.Image(plotted_rgb), n, max_conf)

wandb.log({'test/sample_predictions': predictions_table})
print(f'Logged {len(samples)} sample predictions to W&B.')

## 7 · Lưu artifact + copy về Drive

In [ ]:
import shutil

# Re-attach to W&B if needed (cell 5/6 may have closed the run)
if wandb.run is None:
    try:
        wandb.init(project=run.project, id=run.id, resume='allow')
    except (NameError, AttributeError):
        wandb.init(project=config['project_name'],
                   name=config['run_name'] + '-artifact',
                   config=config, tags=['artifact'])

# Save artifact in W&B
artifact = wandb.Artifact(
    name='yolov8n-hand-detector', type='model',
    description=f"mAP@0.5={metrics.box.map50:.4f} mAP@0.5:0.95={metrics.box.map:.4f}",
    metadata={
        'mAP50': float(metrics.box.map50),
        'mAP50_95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
        'epochs': config['epochs'],
        'model_variant': config['model_variant'],
    },
)
artifact.add_file(BEST_PT, name='best.pt')
artifact.add_file(LAST_PT, name='last.pt')
wandb.log_artifact(artifact)

# Copy về Drive
shutil.copy(BEST_PT, f'{SAVE_DIR}/yolov8n_hand.pt')
print(f'Saved:\n  Drive: {SAVE_DIR}/yolov8n_hand.pt\n  W&B artifact: {artifact.name}')

wandb.finish()

## 8 · Test pipeline 2-stage end-to-end (sanity check)

Nếu bạn đã train xong classifier (notebook kia) → load cả 2 weights và test 1 ảnh tay làm chữ:

In [ ]:
# Optional — chỉ chạy nếu file classifier tồn tại trong Drive
CLS_PATH = f'{SAVE_DIR}/cnn_resnet18.pt'
if not os.path.exists(CLS_PATH):
    print(f'⚠️  {CLS_PATH} không tồn tại — bỏ qua, chạy notebook classifier trước.')
else:
    from torchvision.transforms import v2 as T
    from torch import nn
    from torchvision.models import resnet18

    # Load classifier
    cls_ckpt = torch.load(CLS_PATH, weights_only=False)
    cls = resnet18(weights=None); cls.fc = nn.Linear(cls.fc.in_features, cls_ckpt['num_classes'])
    cls.load_state_dict(cls_ckpt['model']); cls.eval()
    cls = cls.to('cuda' if torch.cuda.is_available() else 'cpu')
    CLASS_NAMES = cls_ckpt['class_names']
    cls_tf = T.Compose([T.Resize((224, 224), antialias=True),
                        T.ToImage(), T.ToDtype(torch.float32, scale=True),
                        T.Normalize((0.485,0.456,0.406), (0.229,0.224,0.225))])

    # Run 2-stage on a sample
    sample = samples[0]
    res = best.predict(source=sample, conf=0.25, verbose=False)[0]
    if len(res.boxes) == 0:
        print(f'No hand detected in {sample}')
    else:
        # Crop highest-confidence box
        x1, y1, x2, y2 = res.boxes.xyxy[0].cpu().numpy().astype(int)
        img = Image.open(sample).convert('RGB')
        crop = img.crop((x1, y1, x2, y2))
        x = cls_tf(crop).unsqueeze(0).to(cls.parameters().__next__().device)
        with torch.no_grad():
            probs = cls(x).softmax(-1)[0].cpu().numpy()
        top3 = probs.argsort()[::-1][:3]
        print(f'Detection bbox: ({x1},{y1})→({x2},{y2})  conf={float(res.boxes.conf[0]):.2f}')
        print('Classifier top-3:')
        for i in top3:
            print(f'  {CLASS_NAMES[i]}: {probs[i]*100:.1f}%')

## Bước tiếp theo (trên máy local)

Khi đã có cả 2 weights trong Drive (`yolov8n_hand.pt` + `cnn_resnet18.pt`):

```bash
cd CV/Project/signlang
make setup

# Copy weights vào đúng path mà demo.yaml expect
mkdir -p runs/detector/weights runs/classifier_resnet18
cp ~/Drive/MyDrive/signlang/yolov8n_hand.pt runs/detector/weights/best.pt
cp ~/Drive/MyDrive/signlang/cnn_resnet18.pt runs/classifier_resnet18/best.pt

make demo            # OpenCV window — cần webcam
# hoặc
make demo-web        # Gradio @ localhost:7860
```

## So sánh runs trong báo cáo

Mở https://wandb.ai/your-username/signlang-detector — chọn 2+ runs (vd yolov8n vs yolov8s) → click "Compare" → có sẵn:
- Side-by-side mAP curves
- Bảng config khác nhau
- Confusion matrix per run
- Sample prediction tables

Screenshot phần này → đưa vào slide ablation.